In [1]:
# ============================================================
# SECTION 1: IMPORT LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb

print('Libraries loaded!')

Libraries loaded!


In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTarget distribution:\n{train['Irrigation_Need'].value_counts()}")

Train shape: (630000, 21)
Test shape: (270000, 20)

Target distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


In [22]:
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [8]:
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [3]:
# Check missing values
print(f"\nMissing values in train:\n{train.isna().sum()}")
print(f"\nMissing values in test:\n{test.isna().sum()}")


Missing values in train:
id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64

Missing values in test:
id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Typ

In [5]:
# Identify categorical and numerical features
categorical_features = train.select_dtypes(include='object').columns.tolist()
categorical_features.remove('Irrigation_Need')  # Remove target
numerical_features = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_features.remove('id')  # Remove id

print(f"\nCategorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")


Categorical features: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numerical features: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


In [6]:
# Separate features and target
X_train_full = train.drop(['Irrigation_Need', 'id'], axis=1)
y_train = train['Irrigation_Need']
X_test = test.drop('id', axis=1)
test_ids = test['id']

In [7]:
# Encode categorical variables
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    X_train_full[col] = le.fit_transform(X_train_full[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le

# Encode target variable
le_target = LabelEncoder()
y_train_encoded = le_target.fit_transform(y_train)

# Align columns between train and test
X_train_full, X_test = X_train_full.align(X_test, join='left', axis=1, fill_value=0)

print(f"\nEncoded features shape: {X_train_full.shape}")


Encoded features shape: (630000, 19)


In [8]:
X_train_full.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,1,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,4,2,2,1,1,0.82,0,112.16,1
1,0,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,5,3,0,2,3,5.27,1,47.16,3
2,0,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,3,3,0,3,2,8.24,1,110.38,2
3,2,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,5,0,0,0,3,8.32,1,53.85,3
4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,5,2,1,0,3,7.37,0,93.19,3


In [9]:
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_full, y_train_encoded, 
    test_size=0.2, 
    random_state=42,
    stratify=y_train_encoded
)

print(f"Train set: {X_train.shape}, Val set: {X_val.shape}")

Train set: (504000, 19), Val set: (126000, 19)


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [12]:
models = {
    'XGBoost': XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        num_class=3,
        objective='multi:softmax'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ),
}

results = []

print("\n" + "="*60)
print("MODEL TRAINING & EVALUATION")
print("="*60)

for name, model in models.items():
    print(f"\n{name}...")
    
    # Train
    if name == 'XGBoost' or name == 'LightGBM':
        model.fit(X_train, y_train_split)
    else:
        model.fit(X_train_scaled, y_train_split)
    
    # Validate
    if name == 'XGBoost' or name == 'LightGBM':
        y_pred = model.predict(X_val)
    else:
        y_pred = model.predict(X_val_scaled)
    
    balanced_acc = balanced_accuracy_score(y_val, y_pred)
    print(f"{name} Balanced Accuracy: {balanced_acc:.4f}")
    
    results.append({
        'model': name,
        'balanced_acc': balanced_acc,
        'model_obj': model
    })
    
    print(f"Classification Report:\n{classification_report(y_val, y_pred)}")


MODEL TRAINING & EVALUATION

XGBoost...
XGBoost Balanced Accuracy: 0.9645
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      4202
           1       0.99      0.99      0.99     73983
           2       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000


LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005311 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2694
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0

In [13]:
best_result = max(results, key=lambda x: x['balanced_acc'])
best_model = best_result['model_obj']
best_model_name = best_result['model']

print(f"\nBest Model: {best_model_name} with Balanced Accuracy: {best_result['balanced_acc']:.4f}")


Best Model: LightGBM with Balanced Accuracy: 0.9649


In [14]:
print("\nGenerating predictions on test set...")

if best_model_name in ['XGBoost', 'LightGBM']:
    y_pred_test = best_model.predict(X_test)
else:
    y_pred_test = best_model.predict(X_test_scaled)

# Decode predictions back to original class names
y_pred_test_labels = le_target.inverse_transform(y_pred_test)


Generating predictions on test set...


In [15]:
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': y_pred_test_labels
})

print("\nSubmission preview:")
print(submission.head(10))

# Save submission
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved to 'submission.csv'")


Submission preview:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007          Medium
8  630008            High
9  630009             Low

Submission saved to 'submission.csv'
